In [61]:
import scanpy as sc
import numpy as np


path_to_data = "../data/pp_data-24-11-10-01/data.h5ad"
adata = sc.read_h5ad(path_to_data)

print("Initial shape:", adata.X.shape)

# Replace NaN with 0 (or another value like the column median)
adata.X = np.nan_to_num(adata.X, nan=0)

# Filter genes and cells
sc.pp.filter_genes(adata, min_cells=1)
sc.pp.filter_cells(adata, min_genes=1)
print("Post-filtering shape:", adata.X.shape)

Initial shape: (6886, 19690)
Post-filtering shape: (6822, 19576)


In [62]:
import numpy as np

# Remove cells with zero total counts, ignoring NaN during summation
valid_cells = np.nansum(adata.X, axis=1) > 0  # Ignore NaNs while summing
adata = adata[valid_cells]
print("After removing zero total count cells:", adata.X.shape)

After removing zero total count cells: (6512, 19576)


In [63]:
# Normalize total counts
sc.pp.normalize_total(adata, target_sum=1e10)
print("Normalization completed.")

/home/ddalton/miniconda3/envs/scgpt/lib/python3.10/site-packages/scanpy/preprocessing/_normalization.py:207: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


Normalization completed.


In [40]:
adata.X

array([[  2683.96711082, -10462.64197132,  40543.77258761, ...,
             0.        ,      0.        ,      0.        ],
       [  9346.34904113, -10923.30604356,  34214.37798198, ...,
             0.        ,      0.        ,      0.        ],
       [ 13412.75328841,  -7362.40154467,  37092.82070765, ...,
             0.        ,      0.        ,      0.        ],
       ...,
       [     0.        , 716498.46363396, 739123.40775105, ...,
             0.        ,      0.        ,      0.        ],
       [     0.        , 747109.16681356, 730533.134921  , ...,
             0.        ,      0.        ,      0.        ],
       [     0.        , 710998.58566712, 740128.72003205, ...,
             0.        ,      0.        ,      0.        ]])

In [50]:
# Apply ComBat to correct for batch effects
sc.pp.combat(adata, key="batch")  # 'batch' is the column in adata.obs with batch labels

/home/ddalton/miniconda3/envs/scgpt/lib/python3.10/site-packages/scanpy/preprocessing/_combat.py:351: RuntimeWarning: divide by zero encountered in divide
  (abs(g_new - g_old) / g_old).max(), (abs(d_new - d_old) / d_old).max()


In [51]:
adata.X

array([[ 1.36665956e+04,  1.27592881e+06,  1.23595825e+06, ...,
         1.84653870e+02,  1.99618140e+05,  1.20315310e+03],
       [ 2.18436426e+05,  1.26195310e+06,  9.65331053e+05, ...,
         1.84653870e+02,  1.99618140e+05,  1.20315310e+03],
       [ 3.43418289e+05,  1.36998443e+06,  1.08840521e+06, ...,
         1.84653870e+02,  1.99618140e+05,  1.20315310e+03],
       ...,
       [ 5.13936485e+05, -2.52041637e+06, -2.62535108e+04, ...,
         1.84367166e+02,  1.99290205e+05,  1.20136589e+03],
       [ 5.13936485e+05, -1.25416725e+06, -3.35540354e+05, ...,
         1.84367166e+02,  1.99290205e+05,  1.20136589e+03],
       [ 5.13936485e+05, -2.74792554e+06,  9.94206390e+03, ...,
         1.84367166e+02,  1.99290205e+05,  1.20136589e+03]])

In [53]:
# Log-transform the normalized data
sc.pp.log1p(adata)

/home/ddalton/miniconda3/envs/scgpt/lib/python3.10/site-packages/scanpy/preprocessing/_simple.py:384: RuntimeWarning: invalid value encountered in log1p
  np.log1p(X, out=X)


In [65]:
adata.X

array([[ 7.89542376,         nan, 10.61016214, ...,  0.        ,
         0.        ,  0.        ],
       [ 9.14284806,         nan, 10.44043047, ...,  0.        ,
         0.        ,  0.        ],
       [ 9.50403582,         nan, 10.52120568, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 0.        , 13.48213278, 13.51322153, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        , 13.52396793, 13.50153124, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        , 13.47442713, 13.51458075, ...,  0.        ,
         0.        ,  0.        ]])

In [66]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

# Grouped Stratified K-Fold
skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# Features (gene expression matrix)
X = (
    adata.X if isinstance(adata.X, np.ndarray) else adata.X.A
)  # Convert sparse to dense if needed
X = np.where(np.isnan(X), 0, X)  # Replace NaN with 0

# Target variable (disease)
y = adata.obs["mesh_disease"]

# Group variable (e.g., dataset_id)
groups = adata.obs["batch_id"]

# Initialize results storage
overall_results = {}

# Iterate through each unique disease
for disease in adata.obs["mesh_disease"].unique():
    print(f"Training model for disease: {disease}")

    # Filter data for the current disease (binary classification: disease vs. non-disease)
    y_binary = (y == disease).astype(int)  # 1 for the current disease, 0 for others

    # Initialize results storage for this disease
    disease_results = []

    # Perform Grouped Stratified K-Fold
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y_binary, groups)):
        print(f"  Fold {fold + 1}")

        # Split data
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y_binary.iloc[train_idx], y_binary.iloc[test_idx]

        # Initialize the MLP classifier
        model = MLPClassifier(
            hidden_layer_sizes=(128, 64),  # Two hidden layers with 128 and 64 neurons
            max_iter=200,  # Maximum number of iterations
            random_state=42,
        )

        # Train the model
        model.fit(X_train, y_train)

        # Evaluate the model
        y_pred = model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        f1_val = f1_score(
            y_test, y_pred, average="binary"
        )  # Evaluate binary classification

        print(f"    Accuracy: {accuracy:.4f}, F1-Score: {f1_val:.4f}")
        disease_results.append((accuracy, f1_val))

        break
    # Store results for this disease
    overall_results[disease] = {
        "mean_accuracy": np.mean([res[0] for res in disease_results]),
        "mean_f1_score": np.mean([res[1] for res in disease_results]),
    }
    break
# Overall performance summary
for disease, metrics in overall_results.items():
    print(f"Disease: {disease}")
    print(f"  Mean Accuracy: {metrics['mean_accuracy']:.4f}")
    print(f"  Mean F1-Score: {metrics['mean_f1_score']:.4f}")

Training model for disease: Dermatomyositis
  Fold 1
    Accuracy: 0.9780, F1-Score: 0.0000
Disease: Dermatomyositis
  Mean Accuracy: 0.9780
  Mean F1-Score: 0.0000


In [60]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

# Grouped Stratified K-Fold
skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# Features (gene expression matrix)
X = (
    adata.X if isinstance(adata.X, np.ndarray) else adata.X.A
)  # Convert sparse to dense if needed
X = np.where(np.isnan(X), 0, X)  # Replace NaN with 0

# Target variable (disease)
y = adata.obs["mesh_disease"]

# Group variable (e.g., dataset_id)
groups = adata.obs["batch_id"]

# Initialize results storage
overall_results = {}

# Iterate through each unique disease
for disease in adata.obs["mesh_disease"].unique():
    print(f"Training model for disease: {disease}")

    # Filter data for the current disease (binary classification: disease vs. non-disease)
    y_binary = (y == disease).astype(int)  # 1 for the current disease, 0 for others

    # Initialize results storage for this disease
    disease_results = []

    # Perform Grouped Stratified K-Fold
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y_binary, groups)):
        print(f"  Fold {fold + 1}")

        # Split data
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y_binary.iloc[train_idx], y_binary.iloc[test_idx]

        # Initialize the MLP classifier
        model = MLPClassifier(
            hidden_layer_sizes=(128, 64),  # Two hidden layers with 128 and 64 neurons
            max_iter=200,  # Maximum number of iterations
            random_state=42,
        )

        # Train the model
        model.fit(X_train, y_train)

        # Evaluate the model
        y_pred = model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        f1_val = f1_score(
            y_test, y_pred, average="binary"
        )  # Evaluate binary classification

        print(f"    Accuracy: {accuracy:.4f}, F1-Score: {f1_val:.4f}")
        disease_results.append((accuracy, f1_val))

        break
    # Store results for this disease
    overall_results[disease] = {
        "mean_accuracy": np.mean([res[0] for res in disease_results]),
        "mean_f1_score": np.mean([res[1] for res in disease_results]),
    }
    break
# Overall performance summary
for disease, metrics in overall_results.items():
    print(f"Disease: {disease}")
    print(f"  Mean Accuracy: {metrics['mean_accuracy']:.4f}")
    print(f"  Mean F1-Score: {metrics['mean_f1_score']:.4f}")

Training model for disease: Dermatomyositis
  Fold 1
    Accuracy: 0.9895, F1-Score: 0.6857
Disease: Dermatomyositis
  Mean Accuracy: 0.9895
  Mean F1-Score: 0.6857
